In [48]:
!pip install requests python-dotenv

In [30]:
import pandas as pd
import requests
import os
from dotenv import load_dotenv

In [31]:
load_dotenv()
API_KEY = os.getenv("OPENWEATHER_API_KEY")


In [32]:
cities = ["Lucknow", "Delhi", "Mumbai", "Chennai", "Shimla"]
all_data = []

In [33]:
for city in cities:
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city},IN&appid={API_KEY}&units=metric"
    data = requests.get(url).json()
    all_data.append(data)

In [34]:
df = pd.json_normalize(all_data).head()
df["weather"]


0    [{'id': 800, 'main': 'Clear', 'description': '...
1    [{'id': 800, 'main': 'Clear', 'description': '...
2    [{'id': 800, 'main': 'Clear', 'description': '...
3    [{'id': 804, 'main': 'Clouds', 'description': ...
4    [{'id': 800, 'main': 'Clear', 'description': '...
Name: weather, dtype: object

In [35]:

df["weather_main"] = df["weather"].str[0].str["main"]

In [36]:
cols = ['dt','name', 'cod',
        'coord.lon', 'coord.lat', 'main.temp',
         'main.humidity',  'wind.speed','rain.1h','weather_main'
       ]
df = df.reindex(columns = cols)
df

,dt,name,cod,coord.lon,coord.lat,main.temp,main.humidity,wind.speed,rain.1h,weather_main
0,1791982653,Lucknow,200,80.9167,26.8500,27.98,60,1.12,NaN,Clear
1,1791982609,Delhi,200,77.2167,28.6667,30.36,45,2.83,NaN,Clear
2,1791982823,Mumbai,200,72.8479,19.0144,29.32,69,6.61,NaN,Clear
3,1791982699,Chennai,200,80.2785,13.0878,28.91,84,4.04,NaN,Clouds
4,1791982915,Shimla,200,77.1667,31.1000,13.13,73,0.54,NaN,Clear


In [37]:
df = df.rename(columns = {"name": "city",
    "main.temp": "temp_c",
    "main.humidity": "humidity",
    "main.pressure": "pressure",
    "wind.speed": "wind_speed",
    "rain.1h": "rain_1h",
    "coord.lat": "lat",
    "coord.lon": "lon",
                    })
df

,dt,city,cod,lon,lat,temp_c,humidity,wind_speed,rain_1h,weather_main
0,1791982653,Lucknow,200,80.9167,26.8500,27.98,60,1.12,NaN,Clear
1,1791982609,Delhi,200,77.2167,28.6667,30.36,45,2.83,NaN,Clear
2,1791982823,Mumbai,200,72.8479,19.0144,29.32,69,6.61,NaN,Clear
3,1791982699,Chennai,200,80.2785,13.0878,28.91,84,4.04,NaN,Clouds
4,1791982915,Shimla,200,77.1667,31.1000,13.13,73,0.54,NaN,Clear


In [38]:
df.isnull().sum()

dt              0
city            0
cod             0
lon             0
lat             0
temp_c          0
humidity        0
wind_speed      0
rain_1h         5
weather_main    0
dtype: int64

In [39]:
df["rain_1h"] = df["rain_1h"].fillna(0)
df["timestamp_utc"] = pd.to_datetime(df["dt"], unit="s")

In [40]:
df

,dt,city,cod,lon,lat,temp_c,humidity,wind_speed,rain_1h,weather_main,timestamp_utc
0,1791982653,Lucknow,200,80.9167,26.8500,27.98,60,1.12,0.0,Clear,2026-10-14 12:57:33
1,1791982609,Delhi,200,77.2167,28.6667,30.36,45,2.83,0.0,Clear,2026-10-14 12:56:49
2,1791982823,Mumbai,200,72.8479,19.0144,29.32,69,6.61,0.0,Clear,2026-10-14 13:00:23
3,1791982699,Chennai,200,80.2785,13.0878,28.91,84,4.04,0.0,Clouds,2026-10-14 12:58:19
4,1791982915,Shimla,200,77.1667,31.1000,13.13,73,0.54,0.0,Clear,2026-10-14 13:01:55


In [41]:
df.drop("dt",axis = 1,inplace = True)

In [42]:
df

,city,cod,lon,lat,temp_c,humidity,wind_speed,rain_1h,weather_main,timestamp_utc
0,Lucknow,200,80.9167,26.8500,27.98,60,1.12,0.0,Clear,2026-10-14 12:57:33
1,Delhi,200,77.2167,28.6667,30.36,45,2.83,0.0,Clear,2026-10-14 12:56:49
2,Mumbai,200,72.8479,19.0144,29.32,69,6.61,0.0,Clear,2026-10-14 13:00:23
3,Chennai,200,80.2785,13.0878,28.91,84,4.04,0.0,Clouds,2026-10-14 12:58:19
4,Shimla,200,77.1667,31.1000,13.13,73,0.54,0.0,Clear,2026-10-14 13:01:55


In [43]:
import sqlite3

conn = sqlite3.connect("climate.db")
pd.read_sql("SELECT * FROM weather", conn)

,city,timestamp_utc,temp_c,humidity,pressure,wind_speed,rain_1h,weather_main,lat,lon
0,Lucknow,2026-10-09 05:18:57,25.37,81,1014,2.50,0.00,Clear,26.8500,80.9167
1,Delhi,2026-10-09 05:11:24,28.51,57,1014,2.13,0.00,Clear,28.6667,77.2167
2,Mumbai,2026-10-09 05:12:42,29.78,64,1014,3.19,0.00,Clear,19.0144,72.8479
3,Chennai,2026-10-09 05:12:58,33.42,68,1014,3.72,0.00,Clouds,13.0878,80.2785
4,Shimla,2026-10-09 05:21:18,16.69,71,1016,1.39,0.00,Clouds,31.1000,77.1667
5,Lucknow,2026-10-09 12:00:01,26.22,71,1011,2.46,0.00,Clouds,26.8500,80.9167
6,Delhi,2026-10-09 11:58:53,32.84,34,1009,2.37,0.00,Clear,28.6667,77.2167
7,Mumbai,2026-10-09 12:00:19,30.24,65,1010,5.96,0.00,Clear,19.0144,72.8479
8,Chennai,2026-10-09 11:54:12,30.61,72,1010,5.49,0.00,Clouds,13.0878,80.2785
9,Shimla,2026-10-09 12:02:48,14.18,73,1014,0.92,0.00,Clouds,31.1000,77.1667


In [160]:
# conn = sqlite3.connect("climate.db")
# conn.execute("DELETE FROM weather WHERE city IS NULL")
# conn.commit()
# pd.read_sql("SELECT * FROM weather", conn)

In [161]:
# conn.execute("""
# DELETE FROM weather
# WHERE rowid NOT IN (
#     SELECT MIN(rowid) FROM weather GROUP BY city, timestamp_utc
# )
# """)
# conn.commit()

# pd.read_sql("SELECT * FROM weather", conn)

In [44]:
conn = sqlite3.connect("climate.db")
pd.read_sql("SELECT * FROM weather", conn)

,city,timestamp_utc,temp_c,humidity,pressure,wind_speed,rain_1h,weather_main,lat,lon
0,Lucknow,2026-10-09 05:18:57,25.37,81,1014,2.50,0.00,Clear,26.8500,80.9167
1,Delhi,2026-10-09 05:11:24,28.51,57,1014,2.13,0.00,Clear,28.6667,77.2167
2,Mumbai,2026-10-09 05:12:42,29.78,64,1014,3.19,0.00,Clear,19.0144,72.8479
3,Chennai,2026-10-09 05:12:58,33.42,68,1014,3.72,0.00,Clouds,13.0878,80.2785
4,Shimla,2026-10-09 05:21:18,16.69,71,1016,1.39,0.00,Clouds,31.1000,77.1667
5,Lucknow,2026-10-09 12:00:01,26.22,71,1011,2.46,0.00,Clouds,26.8500,80.9167
6,Delhi,2026-10-09 11:58:53,32.84,34,1009,2.37,0.00,Clear,28.6667,77.2167
7,Mumbai,2026-10-09 12:00:19,30.24,65,1010,5.96,0.00,Clear,19.0144,72.8479
8,Chennai,2026-10-09 11:54:12,30.61,72,1010,5.49,0.00,Clouds,13.0878,80.2785
9,Shimla,2026-10-09 12:02:48,14.18,73,1014,0.92,0.00,Clouds,31.1000,77.1667


In [45]:
pd.read_sql("SELECT * FROM weather ORDER BY timestamp_utc DESC", conn)

,city,timestamp_utc,temp_c,humidity,pressure,wind_speed,rain_1h,weather_main,lat,lon
0,Shimla,2026-10-14 13:01:55,13.13,73,1018,0.54,0.00,Clear,31.1000,77.1667
1,Mumbai,2026-10-14 13:00:23,29.32,69,1010,6.61,0.00,Clear,19.0144,72.8479
2,Chennai,2026-10-14 12:58:19,28.91,84,1011,4.04,0.00,Clouds,13.0878,80.2785
3,Lucknow,2026-10-14 12:57:33,27.98,60,1013,1.12,0.00,Clear,26.8500,80.9167
4,Delhi,2026-10-14 12:56:49,30.36,45,1013,2.83,0.00,Clear,28.6667,77.2167
5,Shimla,2026-10-13 12:30:04,12.93,76,1019,0.69,0.16,Rain,31.1000,77.1667
6,Mumbai,2026-10-13 12:29:27,30.14,63,1011,5.05,0.00,Clear,19.0144,72.8479
7,Lucknow,2026-10-13 12:26:26,28.96,52,1014,1.41,0.00,Clear,26.8500,80.9167
8,Chennai,2026-10-13 12:25:01,31.19,69,1010,6.85,0.00,Clouds,13.0878,80.2785
9,Delhi,2026-10-13 12:20:48,29.11,46,1013,1.75,0.00,Clear,28.6667,77.2167


In [46]:
weather_data = pd.read_sql("SELECT * FROM weather ORDER BY timestamp_utc",conn)
weather_data.to_csv("weather_data.csv",index = False)
weather_data.shape

(35, 10)

In [47]:
weather_data

,city,timestamp_utc,temp_c,humidity,pressure,wind_speed,rain_1h,weather_main,lat,lon
0,Delhi,2026-10-09 05:11:24,28.51,57,1014,2.13,0.00,Clear,28.6667,77.2167
1,Mumbai,2026-10-09 05:12:42,29.78,64,1014,3.19,0.00,Clear,19.0144,72.8479
2,Chennai,2026-10-09 05:12:58,33.42,68,1014,3.72,0.00,Clouds,13.0878,80.2785
3,Lucknow,2026-10-09 05:18:57,25.37,81,1014,2.50,0.00,Clear,26.8500,80.9167
4,Shimla,2026-10-09 05:21:18,16.69,71,1016,1.39,0.00,Clouds,31.1000,77.1667
5,Chennai,2026-10-09 11:54:12,30.61,72,1010,5.49,0.00,Clouds,13.0878,80.2785
6,Delhi,2026-10-09 11:58:53,32.84,34,1009,2.37,0.00,Clear,28.6667,77.2167
7,Lucknow,2026-10-09 12:00:01,26.22,71,1011,2.46,0.00,Clouds,26.8500,80.9167
8,Mumbai,2026-10-09 12:00:19,30.24,65,1010,5.96,0.00,Clear,19.0144,72.8479
9,Shimla,2026-10-09 12:02:48,14.18,73,1014,0.92,0.00,Clouds,31.1000,77.1667
